# 04d LLM pipeline 3: Tool Use (active data retrieval)

Idea: the LLM only receives the task context and a description of the available
tools. It decides itself which data to retrieve and can call several tools
iteratively before writing an explanation.

Available tools:

| Tool | Description |
|------|-------------|
| get_feature_schema | Metadata for all features |
| get_feature_importance | Global feature importance |
| get_prediction | Prediction for a custom feature combination |
| get_shap_values | Local contributions for a test instance |
| get_partial_dependence | Partial dependence curve for a feature |
| get_feature_value_context | Position of a feature value in the training set (percentile) |
| get_similar_instances | k most similar training instances |
| get_counterfactual_prediction | What if prediction with changed features |

In [ ]:
from __future__ import annotations

import sys, json, time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import joblib

from utils import (
    INSTANCE_IDS,
    MODELS_DIR, RESULTS_DIR, PROMPTS_DIR,
)
from utils.data import load_train_test
from utils.tools import ToolBox, TOOL_DEFINITIONS
from utils.llm import DEFAULT_MODEL, _get_client, _with_retry, strip_scratchpad

LOSS_KEY   = 'poisson_log'
MODEL      = DEFAULT_MODEL
MAX_TOKENS = 2048

# n=20 validity run: the 10 instances, 1 generation, real time (Tool Use is not
# batchable, it is a client side tool loop). Output in results/pipeline06/.
GEN_INSTANCE_IDS = INSTANCE_IDS
N_GEN            = 1
OUT_DIR          = RESULTS_DIR / 'pipeline06'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'LLM model:     {MODEL}')
print(f'Instances:     {len(GEN_INSTANCE_IDS)} x {N_GEN} generation, real time')
print(f'Output:        {OUT_DIR}')

## 1. Load data and models

In [ ]:
X_train, y_train, X_test, y_test = load_train_test()
from utils.models import load_models
xgb, ebm = load_models(LOSS_KEY)
print(f'X_test: {X_test.shape} | Tools: {[t["name"] for t in TOOL_DEFINITIONS]}')

## 2. System prompt and Tool Use loop

In [ ]:
SYSTEM_PROMPT = (PROMPTS_DIR / "pipeline_06_tooluse.md").read_text()


def tool_use_loop(
    client,
    toolbox: "ToolBox",
    user_message: str,
    system: str,
    max_rounds: int = 10,
) -> tuple[str, list[dict], int, int, str]:
    """Run the Tool Use loop.
    Returns: (final text, tool_calls_log, input_tokens, output_tokens, stop_reason)
    stop_reason: 'end_turn' | 'max_tokens' | 'max_rounds' | other API reason."""
    messages = [{"role": "user", "content": user_message}]
    total_in, total_out = 0, 0
    last_text = ""
    final_stop_reason = "max_rounds"

    for round_num in range(max_rounds):
        response = _with_retry(
            client.messages.create,
            model=MODEL,
            max_tokens=MAX_TOKENS,
            system=[
                {"type": "text", "text": system,
                 "cache_control": {"type": "ephemeral"}}
            ],
            tools=TOOL_DEFINITIONS,
            messages=messages,
        )
        usage = response.usage
        total_in  += usage.input_tokens
        total_out += usage.output_tokens

        messages.append({"role": "assistant", "content": response.content})

        candidate = next(
            (b.text for b in response.content if hasattr(b, "text") and b.text), ""
        )
        if candidate:
            last_text = candidate

        if response.stop_reason == "end_turn":
            return last_text, toolbox.call_log, total_in, total_out, "end_turn"

        if response.stop_reason != "tool_use":
            final_stop_reason = response.stop_reason
            break

        tool_results = []
        for block in response.content:
            if block.type != "tool_use":
                continue
            result = toolbox.dispatch(block.name, block.input)
            tool_results.append({
                "type":        "tool_result",
                "tool_use_id": block.id,
                "content":     json.dumps(result, ensure_ascii=False),
            })
            print(f"    [{round_num+1}] {block.name}({list(block.input.keys())}) -> ok")

        messages.append({"role": "user", "content": tool_results})

    if final_stop_reason == "max_rounds":
        print(f"  [WARN] max_rounds={max_rounds} reached, using the last partial text.")
    else:
        print(f"  [WARN] stop_reason='{final_stop_reason}', using the last partial text.")
    return last_text or "[No answer]", toolbox.call_log, total_in, total_out, final_stop_reason


print("Tool Use loop defined.")

## 3. Task prompt per instance

In [ ]:
from utils.explanations import (
    WEEKDAY_NAMES, MONTH_NAMES, WEATHER_NAMES,
    TEMP_FACTOR, HUM_FACTOR, WIND_FACTOR,
)


def build_task_prompt(model_name: str, instance_id: int) -> str:
    row = X_test.iloc[instance_id]
    y   = float(y_test.iloc[instance_id])

    temp_c   = float(row["temp"])      * TEMP_FACTOR
    hum_pct  = float(row["hum"])       * HUM_FACTOR
    wind_kmh = float(row["windspeed"]) * WIND_FACTOR

    lines = [
        f"I want to understand the prediction for test instance {instance_id}.",
        f"",
        f"Situation (denormalised values for orientation):",
        f"  Model:               {model_name.upper()}",
        f"  Time:                {int(row['hr']):02d}:00",
        f"  Weekday:             {WEEKDAY_NAMES[int(row['weekday'])]}",
        f"  Month:               {MONTH_NAMES[int(row['mnth'])]}",
        f"  Year:                {'2012' if int(row['yr']) == 1 else '2011'}",
        f"  Weather:             {WEATHER_NAMES.get(int(row['weathersit']), int(row['weathersit']))}",
        f"  Temperature:         ~{temp_c:.1f} C",
        f"  Humidity:            {hum_pct:.0f} %",
        f"  Wind speed:          {wind_kmh:.1f} km/h",
        f"  Holiday:             {'yes' if int(row['holiday']) == 1 else 'no'}",
        f"  Actual rentals:      {int(y)} bikes",
        f"",
        f"Please use at least 4 of the available tools to analyse the prediction",
        f"thoroughly. In particular call get_shap_values and",
        f"get_feature_value_context for the most important drivers.",
        f"Then write an understandable explanation in English.",
    ]
    return "\n".join(lines)


# Example prompt
print(build_task_prompt("xgb", INSTANCE_IDS[0]))

## 4. LLM calls

In [ ]:
from utils import run_resumable_generation, build_generation_record

client  = _get_client()
model_objs = {"xgb": xgb, "ebm": ebm}

# Resume/persistence contract centralised in utils.run_resumable_generation
# (skip if exists, idempotent, lossless, tested in tests/test_generation_loop.py).
# generate returns None on an error, so the instance stays open and is retried on
# the next run instead of being persisted in a broken state.
# For N_GEN > 1 each instance is called N times (gen_idx 0..N-1) with its own files.
# A fresh ToolBox per call gives its own call_log per generation.
# The record schema comes from build_generation_record (Tool Use: no cache or
# prediction, with stop_reason/tool_calls, golden test in test_generation_loop.py).

def generate_tooluse(model_name, iid, gen_idx):
    toolbox = ToolBox(model_objs[model_name], X_train, X_test, y_test, model_name)
    prompt  = build_task_prompt(model_name, iid)
    system  = SYSTEM_PROMPT.replace('{model_name}', model_name.upper())

    print(f'\n{model_name.upper()} inst={iid} g{gen_idx}')
    t0 = time.time()
    try:
        raw_text, call_log, in_tok, out_tok, stop_reason = tool_use_loop(
            client, toolbox, prompt, system
        )
    except Exception as e:
        print(f'  [ERROR] {type(e).__name__}: {e}, skipping instance.')
        return None
    elapsed = time.time() - t0
    text = strip_scratchpad(raw_text)

    print(f'  -> {len(call_log)} tool calls  '
          f'in={in_tok}  out={out_tok}  t={elapsed:.1f}s  stop={stop_reason}')

    return build_generation_record(
        pipeline="06_tooluse", model_name=model_name, instance_id=iid,
        explanation=text, usage={"input_tokens": in_tok, "output_tokens": out_tok},
        llm_model=MODEL, loss_key=LOSS_KEY, y_true=float(y_test.iloc[iid]),
        elapsed_s=round(elapsed, 2), include_cache=False,
        extra={"stop_reason": stop_reason, "tool_calls": call_log,
               "n_tool_calls": len(call_log)},
    )


results = run_resumable_generation(
    model_names=["xgb", "ebm"],
    instance_ids=GEN_INSTANCE_IDS,
    out_dir=OUT_DIR,
    generate=generate_tooluse,
    n_generations=N_GEN,
)

totals = {
    "in":  sum(r["usage"]["input_tokens"] for r in results),
    "out": sum(r["usage"]["output_tokens"] for r in results),
}
print(f'\nTotal:  input={totals["in"]}  output={totals["out"]}  ({len(results)} units)')

## 5. Example explanation and tool trace

In [ ]:
rec = results[0]  # first XGBoost explanation
sep = '=' * 70
print(sep)
print(f"Model: {rec['xai_model'].upper()}  |  instance: {rec['instance_id']}  "
      f"|  tool calls: {rec['n_tool_calls']}")
print(sep)
print(rec['explanation'])
print()
print('Tool trace:')
for i, call in enumerate(rec['tool_calls']):
    print(f"  [{i+1}] {call['tool']}({list(call['arguments'].keys())})")
    print(f"       -> {call['result_preview'][:120]}")

## 6. Summary

In [ ]:
import pandas as pd

summary = pd.DataFrame([
    {
        'Model':       r['xai_model'].upper(),
        'Instance':    r['instance_id'],
        'y_true':      r['y_true'],
        'Tool calls':  r['n_tool_calls'],
        'Tools used':  ', '.join({c['tool'] for c in r['tool_calls']}),
        'Words':       len(r['explanation'].split()),
        'tok_input':   r['usage']['input_tokens'],
        'tok_output':  r['usage']['output_tokens'],
        'Time (s)':    r['elapsed_s'],
    }
    for r in results
])
display(summary)